# Old SD Hub Analysis

这个 notebook 做两件事：

- 统计不同图上的节点度分布
- 按 hub / normal 节点分组，统计各模型在测试集上的指标


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd()
sys.path.append(str(REPO_ROOT / 'benchmark' / 'eval'))

from old_sd_analysis_utils import (
    DEFAULT_SD_DIR,
    DEFAULT_SD_PHYS_DIR,
    build_graph_matrices,
    compute_degree_catalog,
    evaluate_run_groups,
    load_dataset_flow,
    load_sensor_ids,
)

sns.set_theme(style='whitegrid')


In [ ]:
# Config
SD_DIR = DEFAULT_SD_DIR
SD_PHYS_DIR = DEFAULT_SD_PHYS_DIR
HUB_RATIO = 0.15
HORIZONS = [3, 6, 12]

# 请按需替换为你实际的 checkpoint 目录
RUNS = [
    ('GWNet', 'distthre', REPO_ROOT / 'BasicTS/checkpoints/GraphWaveNet/SD_original_fixed_100_12_12/50591dd61201a9e7affed49ee0e6d8d1'),
    ('GWNet', 'phys_bidir', REPO_ROOT / 'BasicTS/checkpoints/GraphWaveNet/SD_phys_bidir_100_12_12/956b8f432dc13074ee4743587c9f41e7'),
    ('GWNet', 'identity', REPO_ROOT / 'BasicTS/checkpoints/GraphWaveNet/SD_identity_100_12_12/ee9e8df54b1075e941cdf7dcd9ee79b4'),
    ('DCRNN', 'distthre', REPO_ROOT / 'BasicTS/checkpoints/DCRNN/SD_original_100_12_12/5c795d484ae427ad21a1acf6b3db3665'),
    ('DCRNN', 'phys_dir', REPO_ROOT / 'BasicTS/checkpoints/DCRNN/SD_phys_directed_100_12_12/4e8ac762bd7a875fe10f044f5c6e4934'),
    ('DCRNN', 'identity', REPO_ROOT / 'BasicTS/checkpoints/DCRNN/SD_identity_100_12_12/a94aa7dc438e62616ce6368f2b3b6c31'),
    ('STGCN', 'distthre', REPO_ROOT / 'BasicTS/checkpoints/STGCNChebGraphConv/SD_original_100_12_12/dae9baf01eb26b4ffbe4f2d819b37dac'),
    ('STGCN', 'phys_bidir', REPO_ROOT / 'BasicTS/checkpoints/STGCNChebGraphConv/SD_phys_phys_bidir_100_12_12/662cf22129fd0322635e01374c6b3464'),
    ('STGCN', 'identity', REPO_ROOT / 'BasicTS/checkpoints/STGCNChebGraphConv/SD_identity_100_12_12/31f2ef5393aeac15e149044fdad2fda7'),
]


In [ ]:
flow, desc = load_dataset_flow(SD_DIR)
sensor_ids = load_sensor_ids(SD_DIR, int(desc['num_nodes']))
graphs = build_graph_matrices(SD_DIR, SD_PHYS_DIR)

degree_catalogs = {}
degree_frames = []
for graph_name, adj in graphs.items():
    degree_df = compute_degree_catalog(graph_name, adj, sensor_ids, hub_ratio=HUB_RATIO)
    degree_catalogs[graph_name] = degree_df
    degree_frames.append(degree_df)

degree_catalog = pd.concat(degree_frames, ignore_index=True)
degree_catalog.head()


In [ ]:
degree_summary = (
    degree_catalog.groupby(['graph', 'group'])
    .agg(num_nodes=('node_index', 'count'), mean_degree=('degree', 'mean'), max_degree=('degree', 'max'))
    .reset_index()
)
degree_summary


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(
    data=degree_catalog[degree_catalog['graph'].isin(['distthre', 'phys_dir', 'phys_bidir'])],
    x='degree',
    hue='graph',
    element='step',
    stat='count',
    common_norm=False,
)
plt.title('Degree Distribution on Old SD Graphs')
plt.show()


In [ ]:
group_metric_frames = []
for model_name, experiment, ckpt_dir in RUNS:
    group_metric_frames.append(
        evaluate_run_groups(
            model_name=model_name,
            experiment=experiment,
            ckpt_dir=ckpt_dir,
            degree_catalogs=degree_catalogs,
            num_nodes=int(desc['num_nodes']),
            output_len=int(desc['regular_settings']['OUTPUT_LEN']),
            null_val=float(desc['regular_settings'].get('NULL_VAL', 0.0)),
            horizons=HORIZONS,
        )
    )

group_metrics = pd.concat(group_metric_frames, ignore_index=True)
group_metrics.head()


In [ ]:
pivot = group_metrics.pivot_table(
    index=['model', 'experiment', 'horizon'],
    columns='group',
    values=['MAE', 'RMSE', 'MAPE'],
)
pivot


In [ ]:
overall = group_metrics[group_metrics['horizon'] == 'overall'].copy()
plt.figure(figsize=(12, 5))
sns.barplot(data=overall, x='experiment', y='MAE', hue='group')
plt.xticks(rotation=30)
plt.title('Hub vs Normal Node MAE (overall)')
plt.show()
